# Demo — Kafka Producer

A *minimal* working example. No TODOs, no exercise. Just run each cell from top to bottom and observe what happens. Use it as a reference card while you work on the real exercises.

**The big picture:** a producer is a Python program that sends messages into a Redpanda topic. Redpanda stores those messages durably and lets any number of consumers read them later, at their own pace.

## 1. Connect to the broker

A *broker* is the server program that stores messages. We give the producer one address (`bootstrap.servers`) — the broker then tells the producer about all other brokers in the cluster automatically. In our Codespace there is only one broker, reachable on the docker-internal address `redpanda:29092`.

In [ ]:
from confluent_kafka import Producer
import json, time

BROKER = 'redpanda:29092'

producer = Producer({'bootstrap.servers': BROKER})
print(f'Producer connected to {BROKER}')

## 2. Build the event

Three things make up a Kafka message:

- **Topic** — the *category* of the data (`strom` here for electricity).
  Topics are pre-existing buckets; producers write to them, consumers   read from them.
- **Key** — used by Kafka to decide which *partition* the message lands   in. Same key → always same partition → ordering preserved per key.
- **Value** — the actual payload. We use JSON because it's simple and   human-readable.

In [ ]:
topic = 'strom'
key   = 'haus_a'                          # determines the partition
value = json.dumps({                      # the actual payload
    'haus':      'haus_a',
    'wert':      42.5,
    'einheit':   'kWh',
    'timestamp': time.time(),
})

print(f'Topic: {topic}')
print(f'Key:   {key}')
print(f'Value: {value}')

## 3. Send the event

`producer.produce(...)` only *queues* the message in memory — it returns immediately. Two consequences:

1. We pass a *callback* (`on_delivery`) that Kafka calls once the broker    acknowledges (or rejects) the message.
2. We must call `producer.flush()` to wait until the in-memory queue is    actually delivered. Without `flush`, the program could exit before    anything is sent.

Both `key` and `value` must be **bytes**, not strings — that's why we call `.encode()`. Kafka itself doesn't care about the payload format; JSON is just our convention.

In [ ]:
def on_delivery(err, msg):
    if err:
        print(f'Failed: {err}')
    else:
        print(f'Delivered -> topic={msg.topic()} '
              f'partition={msg.partition()} offset={msg.offset()}')

producer.produce(topic, key=key.encode(), value=value.encode(),
                 callback=on_delivery)
producer.flush()   # blocks until the broker confirms (or times out)
print('Done.')

## What you should see

- A line like `Delivered -> topic=strom partition=0 offset=0` (the   offset increases each time you re-run).
- The same message in **Redpanda Console** (port `8080` in *PORTS*) →   *Topics* → `strom` → *Messages*.

Every Kafka message has a stable **(partition, offset)** position. Together they form a unique address inside the topic.